# 02 - Entrenamiento MLP final

Este notebook contiene la definicion completa del modelo MLP-VaR final, sus features, la funcion de perdida, el entrenamiento rolling y el guardado de predicciones/tablas.


## Metricas de backtesting

Esta celda define las funciones comunes de evaluacion: UC, CC, DQ, tick loss, Winkler score y etiquetas PASS/FAIL.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2


ALPHA = 0.99
P_EXC = 1.0 - ALPHA
SIG = 0.05


def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    mask = np.isfinite(real_losses) & np.isfinite(var_preds)
    hits = (real_losses[mask] > var_preds[mask]).astype(int)
    return real_losses[mask], var_preds[mask], hits


def tick_loss(real_losses, var_preds, alpha=ALPHA):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1.0 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=ALPHA):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=ALPHA):
    _, _, hits = hit_series(real_losses, var_preds)
    n_obs = len(hits)
    n_exc = int(hits.sum())
    p = 1.0 - alpha
    if n_obs == 0 or n_exc == 0 or n_exc == n_obs:
        return np.nan
    p_hat = n_exc / n_obs
    lr_uc = -2.0 * np.log(
        ((1.0 - p) ** (n_obs - n_exc) * p**n_exc)
        / ((1.0 - p_hat) ** (n_obs - n_exc) * p_hat**n_exc)
    )
    return float(1.0 - chi2.cdf(lr_uc, df=1))


def christoffersen_cc_test(real_losses, var_preds, alpha=ALPHA):
    _, _, hits = hit_series(real_losses, var_preds)
    n00 = n01 = n10 = n11 = 0
    for t in range(1, len(hits)):
        pair = (hits[t - 1], hits[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1

    if (n00 + n01) == 0 or (n10 + n11) == 0:
        return np.nan

    pi01 = n01 / (n00 + n01)
    pi11 = n11 / (n10 + n11)
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)
    l0 = ((1.0 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
    l1 = ((1.0 - pi01) ** n00) * (pi01**n01) * ((1.0 - pi11) ** n10) * (pi11**n11)
    if l0 <= 0 or l1 <= 0:
        return np.nan

    lr_ind = -2.0 * np.log(l0 / l1)
    p_uc = kupiec_test(real_losses, var_preds, alpha=alpha)
    if not np.isfinite(p_uc):
        return np.nan
    lr_uc = chi2.isf(p_uc, df=1)
    return float(1.0 - chi2.cdf(lr_uc + lr_ind, df=2))


def dq_test(real_losses, var_preds, alpha=ALPHA, lags=4):
    _, _, hits = hit_series(real_losses, var_preds)
    n_obs = len(hits)
    p = 1.0 - alpha
    if n_obs <= lags + 5:
        return np.nan

    hit = hits - p
    y = hit[lags:]
    x = np.column_stack([np.ones(len(y))] + [hit[lags - j : n_obs - j] for j in range(1, lags + 1)])
    xtx = x.T @ x
    if np.linalg.matrix_rank(xtx) < xtx.shape[0]:
        return np.nan

    beta = np.linalg.inv(xtx) @ x.T @ y
    resid = y - x @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return np.nan

    dq = float((beta.T @ xtx @ beta) / sigma2)
    return float(1.0 - chi2.cdf(dq, df=x.shape[1]))


def evaluate_var(df: pd.DataFrame, model: str, var_col: str, alpha=ALPHA) -> dict:
    real = df["loss_real"].to_numpy()
    var = df[var_col].to_numpy()
    _, _, hits = hit_series(real, var)
    n_obs = len(hits)
    n_exc = int(hits.sum())
    df_2020 = df.loc[df.index.year == 2020]

    return {
        "Model": model,
        "From": df.index.min().date().isoformat(),
        "To": df.index.max().date().isoformat(),
        "T": n_obs,
        "Expected Viol.": (1.0 - alpha) * n_obs,
        "Violations": n_exc,
        "Violation Rate": n_exc / n_obs,
        "VR Gap": abs(n_exc / n_obs - (1.0 - alpha)),
        "Coverage Bias": n_exc / n_obs - (1.0 - alpha),
        "Tick Loss": tick_loss(real, var, alpha=alpha),
        "Winkler Score": winkler_score(real, var, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var)),
        "Max VaR Level": float(np.nanmax(var)),
        "Min VaR Level": float(np.nanmin(var)),
        "Mean VaR 2020": float(df_2020[var_col].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020[var_col].max()) if len(df_2020) else np.nan,
        "n_negative_var": int(np.sum(var < 0)),
        "p_UC": kupiec_test(real, var, alpha=alpha),
        "p_CC": christoffersen_cc_test(real, var, alpha=alpha),
        "p_DQ": dq_test(real, var, alpha=alpha, lags=4),
    }


def add_test_labels(row: dict, sig=SIG) -> dict:
    row = dict(row)
    row["UC Test"] = "PASS" if row["p_UC"] > sig else "FAIL"
    row["CC Test"] = "PASS" if row["p_CC"] > sig else "FAIL"
    row["DQ Test"] = "PASS" if row["p_DQ"] > sig else "FAIL"
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return row


## Modelo MLP-VaR final

Esta celda define las features de presion bajista, la loss pinball, la arquitectura Softplus SiLU LayerNorm y el entrenamiento rolling.


In [ ]:
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
PREDICTIONS = RESULTS / "predictions"
TABLES = RESULTS / "tables"
DATA.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
PREDICTIONS.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)

ALPHA = 0.99
PINBALL_WEIGHT = 1.0
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42
EVAL_START = "2010-01-01"
EVAL_END = "2024-12-31"

PRED_FILE = PREDICTIONS / "mlp_var_predictions.csv"
SUMMARY_FILE = TABLES / "mlp_var_summary_2010_2024.csv"
YEARLY_FILE = TABLES / "mlp_var_yearly_2010_2024.csv"
VIOLATIONS_FILE = TABLES / "mlp_var_violations_2010_2024.csv"


def add_downside_pressure_features(dataset: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    dataset = dataset.copy().sort_index()
    eps = 1e-8
    rp = dataset["rp_lag_0"]
    abs_r = rp.abs()
    downside = np.maximum(-rp, 0.0)

    vol_2 = rp.rolling(2).std(ddof=0)
    vol_3 = rp.rolling(3).std(ddof=0)
    vol_5 = rp.rolling(5).std(ddof=0)
    vol_10 = rp.rolling(10).std(ddof=0)
    vol_20_realized = rp.rolling(20).std(ddof=0)
    vol_20_ref = dataset["vol_20"].abs()
    vol_60_ref = dataset["vol_60"].abs()

    dataset["vol_5_ratio"] = vol_5 / (vol_20_realized + eps)
    dataset["vol_10_ratio"] = vol_10 / (vol_20_realized + eps)
    dataset["shock_1"] = abs_r / (vol_20_realized + eps)
    dataset["shock_5"] = abs_r.rolling(5).max() / (vol_20_realized + eps)
    dataset["vol_20_ratio_60"] = vol_20_ref / (vol_60_ref + eps)
    dataset["vol_20_ratio_252"] = vol_20_ref / (vol_20_ref.rolling(252).median() + eps)
    dataset["abs_ret_sum_5_ratio"] = abs_r.rolling(5).sum() / (vol_20_realized + eps)
    dataset["abs_ret_sum_10_ratio"] = abs_r.rolling(10).sum() / (vol_20_realized + eps)
    dataset["max_abs_ret_10_ratio"] = abs_r.rolling(10).max() / (vol_20_realized + eps)

    dataset["downside_1_ratio"] = downside / (vol_20_realized + eps)
    dataset["downside_sum_3_ratio"] = downside.rolling(3).sum() / (vol_20_realized + eps)
    dataset["downside_sum_5_ratio"] = downside.rolling(5).sum() / (vol_20_realized + eps)
    dataset["downside_sum_10_ratio"] = downside.rolling(10).sum() / (vol_20_realized + eps)
    dataset["downside_max_5_ratio"] = downside.rolling(5).max() / (vol_20_realized + eps)
    dataset["downside_count_5"] = (downside > 0).rolling(5).sum()
    dataset["downside_count_10"] = (downside > 0).rolling(10).sum()
    dataset["vol_2_ratio"] = vol_2 / (vol_20_realized + eps)
    dataset["vol_3_ratio"] = vol_3 / (vol_20_realized + eps)
    dataset["vol_jump_5_20"] = vol_5 / (vol_20_realized + eps)

    dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()
    feature_cols = [c for c in dataset.columns if c != "target"]
    if len(feature_cols) != 41:
        raise ValueError(f"Se esperaban 41 features, hay {len(feature_cols)}")
    return dataset, feature_cols


def inverse_softplus(y: float) -> float:
    return math.log(math.expm1(y))


def var_loss(y_true, y_pred, q=ALPHA, weight_exceedance=PINBALL_WEIGHT):
    e = y_true - y_pred
    weight = torch.where(
        e > 0,
        torch.as_tensor(weight_exceedance, dtype=y_pred.dtype, device=y_pred.device),
        torch.ones_like(e),
    )
    pinball = torch.maximum(q * e, (q - 1.0) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplusSiLULayerNorm(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


def train_one_model(X_train, y_train, seed=SEED, max_epochs=200, lr=5e-5, batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)

    split = int(len(X_train) * (1.0 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]

    model = MLPVaRSoftplusSiLULayerNorm(d_in=X_train.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=batch_size, shuffle=False)

    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)
    best_loss = np.inf
    best_state = None
    patience_counter = 0

    for _ in range(max_epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = var_loss(yb, model(xb))
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = var_loss(y_val_t, model(X_val_t)).item()

        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def run_rolling_mlp(dataset: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    torch.set_num_threads(1)
    eval_dates = dataset.loc[EVAL_START:EVAL_END].index
    var_preds = []
    real_targets = []
    dates = []
    current_model = None
    mu_x = sd_x = None

    for i, current_date in enumerate(tqdm(eval_dates, desc="MLP-VaR rolling", dynamic_ncols=True)):
        if i % RETRAIN_EVERY == 0:
            train_end = current_date - pd.Timedelta(days=1)
            train_df = dataset.loc[:train_end].tail(WINDOW)
            if len(train_df) >= 250:
                X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
                y_train = train_df["target"].to_numpy(dtype=np.float32).reshape(-1, 1)
                mu_x = X_train.mean(axis=0, keepdims=True)
                sd_x = X_train.std(axis=0, keepdims=True) + 1e-8
                X_train = (X_train - mu_x) / sd_x
                current_model = train_one_model(X_train, y_train)

        if current_model is None or mu_x is None or sd_x is None:
            continue

        test_df = dataset.loc[[current_date]]
        X_test = (test_df[feature_cols].to_numpy(dtype=np.float32) - mu_x) / sd_x
        y_test = float(test_df["target"].iloc[0])
        current_model.eval()
        with torch.no_grad():
            pred = float(current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0])

        dates.append(current_date)
        var_preds.append(pred)
        real_targets.append(y_test)

    pred_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
    pred_df["viol"] = (pred_df["loss_real"] > pred_df["VaR_pred"]).astype(int)
    pred_df["year"] = pred_df.index.year
    return pred_df


def save_outputs(pred_df: pd.DataFrame) -> None:
    pred_df.to_csv(PRED_FILE)

    summary = add_test_labels(evaluate_var(pred_df, model="MLP-VaR", var_col="VaR_pred"))
    pd.DataFrame([summary]).to_csv(SUMMARY_FILE, index=False)

    yearly_rows = []
    for year, group in pred_df.groupby(pred_df.index.year):
        row = add_test_labels(evaluate_var(group, model="MLP-VaR", var_col="VaR_pred"))
        row["year"] = int(year)
        yearly_rows.append(row)
    pd.DataFrame(yearly_rows).to_csv(YEARLY_FILE, index=False)

    violations = pred_df.loc[pred_df["viol"] == 1, ["VaR_pred", "loss_real", "viol", "year"]].copy()
    violations["exceedance"] = violations["loss_real"] - violations["VaR_pred"]
    violations["exceedance_ratio"] = violations["loss_real"] / violations["VaR_pred"]
    violations.to_csv(VIOLATIONS_FILE)


def main() -> None:
    dataset = pd.read_pickle(DATA / "dataset_tfg.pkl").sort_index()
    dataset, feature_cols = add_downside_pressure_features(dataset)
    pred_df = run_rolling_mlp(dataset, feature_cols)
    save_outputs(pred_df)
    print("Saved MLP outputs to", PRED_FILE)


## Entrenamiento y guardado

Esta celda ejecuta el entrenamiento rolling y guarda predicciones, resumen anual y violaciones en `results/`.


In [ ]:
print("Output bias inicial:", inverse_softplus(0.02))
main()


## Comprobacion de outputs

Esta celda carga los archivos generados y muestra una vista rapida para verificar que el paso termino correctamente.


In [ ]:
predictions = pd.read_csv(PREDICTIONS / "mlp_var_predictions.csv", index_col=0, parse_dates=True)
summary = pd.read_csv(TABLES / "mlp_var_summary_2010_2024.csv")
yearly = pd.read_csv(TABLES / "mlp_var_yearly_2010_2024.csv")
display(predictions.head())
display(summary)
display(yearly)
